# Week 3 – Task 1: RAG over PDFs with ChromaDB

<small>Builds a RAG pipeline: discover PDFs → chunk → embed → store in ChromaDB → retrieve → generate answer via the reused Week 2 OpenRouter client. Tasks 2–5 are not implemented here.</small>


## Reusing the Week 2 Client

<small>Imports `ChatApp.ipynb` as a module via `import_ipynb`, so its `OpenRouterClient` isn't duplicated. Must run from the same folder as `ChatApp.ipynb`, `CONSTANTS.py`, and `.env`.</small>


In [1]:
# Reuse Week 2's OpenRouter client
import import_ipynb
import ChatApp

print("Reused from ChatApp.ipynb:")
print(" - OpenRouterClient  :", ChatApp.OpenRouterClient)
print(" - OpenRouterAPIError:", ChatApp.OpenRouterAPIError)
print(" - load_api_key()    :", ChatApp.load_api_key)
print(" - MODELS            :", ChatApp.MODELS)


Reused from ChatApp.ipynb:
 - OpenRouterClient  : <class 'ChatApp.OpenRouterClient'>
 - OpenRouterAPIError: <class 'ChatApp.OpenRouterAPIError'>
 - load_api_key()    : <function load_api_key at 0x000002683914ED40>
 - MODELS            : {'1': 'meta-llama/llama-3.1-8b-instruct', '2': 'google/gemma-3-27b-it', '3': 'nvidia/nemotron-nano-9b-v2:free'}


## Setup

<small>Extra packages needed: `pypdf`, `sentence-transformers`, `chromadb`, `import-ipynb`. `requests`/`python-dotenv` are reused from `ChatApp.ipynb`.</small>


In [2]:
# %pip install -q pypdf sentence-transformers chromadb import-ipynb

import os
import time
from pathlib import Path
from typing import List, Dict, Any, Optional

from pypdf import PdfReader
import chromadb
from sentence_transformers import SentenceTransformer


## The PDF Collection (`data/`)

<small>Four cheat-sheet PDFs (Python, ML, DL, AI) live in `data/`. No filenames are hardcoded — add/remove PDFs and re-run.</small>

| File | Topic | Author(s) |
|---|---|---|
| `python_cheatsheet.pdf` | Python | Eric Matthes |
| `machine_learning_cheatsheet.pdf` | ML (CS229) | Amidi & Amidi |
| `deep_learning_cheatsheet.pdf` | DL (CS230) | Amidi & Amidi |
| `artificial_intelligence_cheatsheet.pdf` | AI (CS221) | Amidi & Amidi |


## Configuration

<small>All pipeline settings live here — change and re-run from this point down.</small>


In [3]:
DATA_DIR = Path("data")
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
TOP_K = 4

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

CHROMA_PERSIST_DIR = "chroma_store"
COLLECTION_NAME = "week3_pdf_knowledge_base"

GENERATION_MODEL_KEY = "1"
GENERATION_MODEL = ChatApp.MODELS[GENERATION_MODEL_KEY]

print(f"Chunking : {CHUNK_SIZE} chars, {CHUNK_OVERLAP} char overlap")
print(f"Retrieval: top-{TOP_K} chunks")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Generation model (via ChatApp): {GENERATION_MODEL}")


Chunking : 800 chars, 150 char overlap
Retrieval: top-4 chunks
Embedding model: all-MiniLM-L6-v2
Generation model (via ChatApp): meta-llama/llama-3.1-8b-instruct


## Step 1 — Discover PDFs

<small>Scans `data/` for `*.pdf` files at runtime. Raises `FileNotFoundError` if the folder is missing or empty.</small>


In [4]:
def discover_pdfs(data_dir: Path) -> List[Path]:
    """Return a sorted list of PDF paths found in data_dir."""
    if not data_dir.exists():
        raise FileNotFoundError(
            f"Data directory '{data_dir}' does not exist. "
            f"Create it and add at least one PDF file."
        )

    pdf_paths = sorted(data_dir.glob("*.pdf"))

    if not pdf_paths:
        raise FileNotFoundError(
            f"No PDF files found in '{data_dir}'. Add at least one .pdf file."
        )

    return pdf_paths


pdf_paths = discover_pdfs(DATA_DIR)
print(f"Discovered {len(pdf_paths)} PDF(s) in '{DATA_DIR}/':")
for path in pdf_paths:
    print(f"  - {path.name}")


Discovered 4 PDF(s) in 'data/':
  - artificial_intelligence_cheatsheet.pdf
  - deep_learning_cheatsheet.pdf
  - machine_learning_cheatsheet.pdf
  - python_cheatsheet.pdf


## Step 2 — Extract Text (`pypdf`)

<small>Returns one record per page of extractable text, tagged with page number and source file. Unreadable PDFs/pages are skipped, not fatal.</small>


In [5]:
def extract_pdf_pages(pdf_path: Path) -> List[Dict[str, Any]]:
    """Extract text per page as a list of {"text", "page", "source"} dicts."""
    pages: List[Dict[str, Any]] = []

    try:
        reader = PdfReader(str(pdf_path))
    except Exception as exc:
        print(f"  WARNING: could not open '{pdf_path.name}' — skipping. ({exc})")
        return pages

    for page_number, page in enumerate(reader.pages, start=1):
        try:
            page_text = page.extract_text() or ""
        except Exception as exc:
            print(f"  WARNING: failed to extract page {page_number} of "
                  f"'{pdf_path.name}' — skipping that page. ({exc})")
            continue

        page_text = page_text.strip()
        if page_text:
            pages.append({
                "text": page_text,
                "page": page_number,
                "source": pdf_path.name,
            })

    if not pages:
        print(f"  WARNING: no extractable text found in '{pdf_path.name}' "
              f"(it may be a scanned/image-only PDF).")

    return pages


## Step 3 — Chunk Text

<small>Slides a `CHUNK_SIZE`-character window forward with `CHUNK_OVERLAP` shared between consecutive chunks. Errors if overlap ≥ size.</small>


In [6]:
def chunk_text(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    """Split text into overlapping, fixed-size chunks."""
    if chunk_overlap >= chunk_size:
        raise ValueError(
            f"chunk_overlap ({chunk_overlap}) must be smaller than "
            f"chunk_size ({chunk_size})."
        )

    chunks: List[str] = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= text_length:
            break
        start = end - chunk_overlap  # advance, keeping overlap

    return chunks


## Step 4 — Build Chunk Records

<small>Combines Steps 2–3: extracts each PDF, chunks the text, and attaches source/page/id metadata for ChromaDB.</small>


In [7]:
def build_chunk_records(
    pdf_paths: List[Path],
    chunk_size: int,
    chunk_overlap: int,
) -> List[Dict[str, Any]]:
    """Extract + chunk every PDF, attaching id/source/page metadata."""
    records: List[Dict[str, Any]] = []

    for pdf_path in pdf_paths:
        print(f"Processing '{pdf_path.name}'...")
        pages = extract_pdf_pages(pdf_path)

        for page_info in pages:
            page_chunks = chunk_text(page_info["text"], chunk_size, chunk_overlap)

            for chunk_index, chunk in enumerate(page_chunks):
                records.append({
                    "id": f"{pdf_path.stem}_p{page_info['page']}_c{chunk_index}",
                    "text": chunk,
                    "source": page_info["source"],
                    "page": page_info["page"],
                    "chunk_index": chunk_index,
                })

    return records


chunk_records = build_chunk_records(pdf_paths, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"\nBuilt {len(chunk_records)} chunks total.")
per_file_counts = {}
for record in chunk_records:
    per_file_counts[record["source"]] = per_file_counts.get(record["source"], 0) + 1
for source, count in per_file_counts.items():
    print(f"  - {source}: {count} chunks")


Processing 'artificial_intelligence_cheatsheet.pdf'...
Processing 'deep_learning_cheatsheet.pdf'...
Processing 'machine_learning_cheatsheet.pdf'...
Processing 'python_cheatsheet.pdf'...

Built 381 chunks total.
  - artificial_intelligence_cheatsheet.pdf: 82 chunks
  - deep_learning_cheatsheet.pdf: 54 chunks
  - machine_learning_cheatsheet.pdf: 70 chunks
  - python_cheatsheet.pdf: 175 chunks


## Step 5 — Generate Embeddings

<small>Loads `all-MiniLM-L6-v2` and encodes chunk texts into vectors for similarity search.</small>


In [8]:
def load_embedding_model(model_name: str) -> SentenceTransformer:
    """Load a sentence-transformers model; raises RuntimeError on failure."""
    try:
        return SentenceTransformer(model_name)
    except Exception as exc:
        raise RuntimeError(
            f"Failed to load embedding model '{model_name}'. Check your "
            f"internet connection and try again. Original error: {exc}"
        ) from exc


def embed_texts(model: SentenceTransformer, texts: List[str]) -> List[List[float]]:
    """Embed a list of texts, returning plain-list vectors."""
    try:
        embeddings = model.encode(
            texts,
            batch_size=32,
            show_progress_bar=True,
            convert_to_numpy=True,
        )
    except Exception as exc:
        raise RuntimeError(f"Embedding generation failed: {exc}") from exc

    return embeddings.tolist()


embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)

chunk_texts = [record["text"] for record in chunk_records]
t0 = time.time()
chunk_embeddings = embed_texts(embedding_model, chunk_texts)
print(f"Generated {len(chunk_embeddings)} embeddings "
      f"(dimension={len(chunk_embeddings[0])}) in {time.time() - t0:.2f}s")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Generated 381 embeddings (dimension=384) in 12.62s


## Step 6 — Store in ChromaDB

<small>Opens a persistent collection (cosine similarity) and upserts chunks + embeddings + metadata. Deterministic IDs avoid duplicates on re-run.</small>


In [9]:
def get_chroma_collection(persist_dir: str, collection_name: str):
    """Open or create a persistent ChromaDB collection (cosine similarity)."""
    try:
        chroma_client = chromadb.PersistentClient(path=persist_dir)
        collection = chroma_client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"},
        )
    except Exception as exc:
        raise RuntimeError(f"ChromaDB initialization failed: {exc}") from exc

    return collection


def index_chunks(collection, records: List[Dict[str, Any]], embeddings: List[List[float]]) -> None:
    """Upsert chunk text, embeddings, and metadata into ChromaDB."""
    try:
        collection.upsert(
            ids=[record["id"] for record in records],
            embeddings=embeddings,
            documents=[record["text"] for record in records],
            metadatas=[
                {
                    "source": record["source"],
                    "page": record["page"],
                    "chunk_index": record["chunk_index"],
                }
                for record in records
            ],
        )
    except Exception as exc:
        raise RuntimeError(f"Failed to store chunks in ChromaDB: {exc}") from exc


collection = get_chroma_collection(CHROMA_PERSIST_DIR, COLLECTION_NAME)
index_chunks(collection, chunk_records, chunk_embeddings)
print(f"ChromaDB collection '{COLLECTION_NAME}' now holds {collection.count()} chunks.")


ChromaDB collection 'week3_pdf_knowledge_base' now holds 381 chunks.


## Step 7 — Retrieve Top-K Chunks

<small>Embeds the question, queries ChromaDB, and converts distance to similarity (`1 - distance`).</small>


In [10]:
def retrieve_relevant_chunks(
    question: str,
    embedding_model: SentenceTransformer,
    collection,
    top_k: int,
) -> List[Dict[str, Any]]:
    """Return the top_k chunks most similar to the question."""
    try:
        question_embedding = embedding_model.encode(
            [question], convert_to_numpy=True
        ).tolist()
    except Exception as exc:
        raise RuntimeError(f"Failed to embed the question: {exc}") from exc

    try:
        results = collection.query(
            query_embeddings=question_embedding,
            n_results=top_k,
            include=["documents", "metadatas", "distances"],
        )
    except Exception as exc:
        raise RuntimeError(f"ChromaDB query failed: {exc}") from exc

    retrieved_chunks = []
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    for doc_text, metadata, distance in zip(documents, metadatas, distances):
        retrieved_chunks.append({
            "text": doc_text,
            "source": metadata["source"],
            "page": metadata["page"],
            "similarity": 1 - distance,
        })

    return retrieved_chunks


def display_retrieved_chunks(retrieved_chunks: List[Dict[str, Any]]) -> None:
    """Print each chunk's source, page, similarity, and a text preview."""
    if not retrieved_chunks:
        print("No relevant chunks were found in the knowledge base.")
        return

    print(f"Retrieved {len(retrieved_chunks)} relevant chunk(s):\n")
    for rank, chunk in enumerate(retrieved_chunks, start=1):
        preview = chunk["text"][:250] + ("..." if len(chunk["text"]) > 250 else "")
        print(f"[{rank}] source={chunk['source']}  page={chunk['page']}  "
              f"similarity={chunk['similarity']:.3f}")
        print(f"    {preview}\n")


## Step 8 — Generate a Grounded Answer

<small>Builds a prompt from the retrieved context + question, then calls the reused `OpenRouterClient.send_chat()`.</small>


In [11]:
RAG_SYSTEM_PROMPT = (
    "You are a helpful assistant that answers questions using ONLY the "
    "context provided below. Do not use any outside knowledge, and do not "
    "guess. If the answer cannot be found in the provided context, respond "
    "exactly with: 'The information is not available in the provided "
    "context.' Always mention which source/page the answer came from when "
    "you do find it in the context."
)


def build_rag_messages(question: str, retrieved_chunks: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    """Build the chat messages for a retrieval-augmented question."""
    context_blocks = [
        f"[Source: {chunk['source']}, page {chunk['page']}]\n{chunk['text']}"
        for chunk in retrieved_chunks
    ]
    context_text = "\n\n---\n\n".join(context_blocks)

    user_message = (
        f"Context:\n{context_text}\n\n"
        f"Question: {question}\n\n"
        f"Answer the question using only the context above."
    )

    return [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]


def generate_answer(
    client: "ChatApp.OpenRouterClient",
    model: str,
    question: str,
    retrieved_chunks: List[Dict[str, Any]],
) -> "tuple[str, Optional[float]]":
    """Ask the reused OpenRouter client to answer from retrieved context."""
    messages = build_rag_messages(question, retrieved_chunks)

    try:
        result, elapsed = client.send_chat(model, messages)
    except ChatApp.OpenRouterAPIError as exc:
        return f"Could not generate an answer: {exc}", None

    try:
        answer = result["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError):
        return "Received an unexpected response format from the API.", None

    return answer, elapsed


## Step 9 — Put It Together: `ask()`

<small>Single entry point: retrieves chunks, displays them, generates the answer, and prints it.</small>


In [12]:
def ask(question: str, top_k: int = TOP_K) -> str:
    """Answer a question end-to-end using the RAG pipeline."""
    print(f'Question: "{question}"\n')

    retrieved_chunks = retrieve_relevant_chunks(question, embedding_model, collection, top_k)
    display_retrieved_chunks(retrieved_chunks)

    if not retrieved_chunks:
        answer = "The information is not available in the provided context."
        print(f"=== Answer ===\n{answer}")
        return answer

    answer, elapsed = generate_answer(client, GENERATION_MODEL, question, retrieved_chunks)

    print("=== Answer ===")
    if elapsed is not None:
        print(f"(generated by {GENERATION_MODEL} in {elapsed:.2f}s)\n")
    print(answer)

    return answer


## Load API Key & Client (Reused)

<small>Calls `ChatApp.load_api_key()` and `ChatApp.OpenRouterClient(...)`, catching the missing-key case cleanly.</small>


In [13]:
try:
    api_key = ChatApp.load_api_key()
except SystemExit as exc:
    raise RuntimeError(
        "OPENROUTER_API_KEY is missing. Make sure a '.env' file is present "
        "in this same folder before continuing."
    ) from exc

client = ChatApp.OpenRouterClient(api_key)
print("OpenRouter client ready. Using model:", GENERATION_MODEL)


OpenRouter client ready. Using model: meta-llama/llama-3.1-8b-instruct


## Demo

<small>One answerable question, one out-of-scope question (to confirm the decline behavior).</small>


In [14]:
_ = ask("What is gradient descent and how is it used in machine learning?")

Question: "What is gradient descent and how is it used in machine learning?"

Retrieved 4 relevant chunk(s):

[1] source=machine_learning_cheatsheet.pdf  page=2  similarity=0.541
    ) + (1 − y) log(1 − z)
]
Linear regression Logistic regression SVM Neural Network
ÌCost function– The cost functionJ is commonly used to assess the performance of a model,
and is deﬁned with the loss functionL as follows:
J(θ) =
m∑
i=1
L(hθ(x(i)),y (...

[2] source=artificial_intelligence_cheatsheet.pdf  page=3  similarity=0.527
    Neural networks– Neural networks are a class of models that are built with layers. Com-
monly used types of neural networks include convolutional and recurrent neural networks. The
vocabulary around neural networks architectures is described in the ﬁ...

[3] source=deep_learning_cheatsheet.pdf  page=13  similarity=0.511
    h, let alone a normal-sized training set.
Ì Gradient checking – Gradient checking is a method used during the implementation of
the backward pass of a neura

In [15]:
_ = ask("What is the capital of France?")  # not covered by these PDFs — should decline

Question: "What is the capital of France?"

Retrieved 4 relevant chunk(s):

[1] source=python_cheatsheet.pdf  page=1  similarity=0.158
    stored as a string. 
Prompting for a value 
name = input("What's your name? ") 
print("Hello, " + name + "!") 
Prompting for numerical input 
age = input("How old are you? ") 
age = int(age) 
 
pi = input("What's the value of pi? ") 
pi = float(pi)

[2] source=deep_learning_cheatsheet.pdf  page=10  similarity=0.158
    t′> = 1
Remark: the attention scores are commonly used in image captioning and machine translation.
Ì Attention weight – The amount of attention that the outputy<t> should pay to the
activationa<t′> is given byα<t,t′> computed as follows:
α<t,t′> = e...

[3] source=python_cheatsheet.pdf  page=16  similarity=0.149
    le(self): 
        """Test names like David Lee Roth.""" 
        full_name = get_full_name('david', 
                'roth', 'lee') 
        self.assertEqual(full_name, 
                'David Lee Roth') 
 
unittest.main

## Try Your Own Question

<small>Run the cell below to type a question interactively.</small>


In [16]:
user_question = input("Enter your question: ").strip()
if user_question:
    _ = ask(user_question)
else:
    print("No question entered.")

Question: "What are the types of Machine Learning?"

Retrieved 4 relevant chunk(s):

[1] source=machine_learning_cheatsheet.pdf  page=1  similarity=0.501
    CS 229 – Machine Learning https://stanford.edu/~shervine
Super VIP Cheatsheet: Machine Learning
Afshine Amidi and Shervine Amidi
October 6, 2018
Contents
1 Supervised Learning 2
1.1 Introduction to Supervised Learning. . . . . . . . . . . . . . . . ....

[2] source=machine_learning_cheatsheet.pdf  page=1  similarity=0.445
    t Vector Machines . . . . . . . . . . . . . . . . . . . . . . . . 3
1.5 Generative Learning . . . . . . . . . . . . . . . . . . . . . . . . . . . 4
1.5.1 Gaussian Discriminant Analysis . . . . . . . . . . . . . . . . . 4
1.5.2 Naive Bayes . . . . . ....

[3] source=artificial_intelligence_cheatsheet.pdf  page=3  similarity=0.414
    Neural networks– Neural networks are a class of models that are built with layers. Com-
monly used types of neural networks include convolutional and recurrent neural networks. Th

## Summary

<small>Pipeline: `discover_pdfs` → `extract_pdf_pages` → `chunk_text` → `build_chunk_records` → `embed_texts` → `index_chunks` → `retrieve_relevant_chunks` → `generate_answer` → `ask()`. Driven entirely by the config block and whatever PDFs are in `data/`. Tasks 2–5 not implemented — this is the foundation for them.</small>


# Task 2: Comparing Embedding Models on Retrieval Quality

<small>Reuses every function and object from Task 1 (`chunk_records`, `client`, `GENERATION_MODEL`, `retrieve_relevant_chunks`, `generate_answer`, etc.) — nothing from Task 1 is redefined or duplicated. Only the embedding model changes between the two ChromaDB collections built below.</small>


## Task 2 Configuration

<small>Two embedding models are compared on the exact same chunks, using the same evaluation questions.</small>


In [17]:
import pandas as pd

# Two embedding models to compare (small/fast vs. larger/more accurate)
EMBEDDING_MODELS_TO_COMPARE = ["all-MiniLM-L6-v2", "all-mpnet-base-v2"]

# Same evaluation questions asked against both collections
EVAL_QUESTIONS = [
    "What is gradient descent and how is it used in machine learning?",
    "What is backpropagation in a neural network?",
    "What is the difference between supervised and unsupervised learning?",
]

print("Comparing embedding models:", EMBEDDING_MODELS_TO_COMPARE)
print(f"Evaluating {len(EVAL_QUESTIONS)} question(s) against each.")


Comparing embedding models: ['all-MiniLM-L6-v2', 'all-mpnet-base-v2']
Evaluating 3 question(s) against each.


## Step 1 — Build One ChromaDB Collection per Embedding Model

<small>Reuses `load_embedding_model`, `embed_texts`, `get_chroma_collection`, and `index_chunks` from Task 1. The same `chunk_records` (already extracted and chunked in Task 1) are embedded and indexed once per model, into separate, uniquely-named collections — so the only variable between them is the embedding model. Embedding and indexing time are recorded for the speed comparison in Step 4.</small>


In [18]:
def build_and_time_collection(
    model_name: str,
    chunk_records: List[Dict[str, Any]],
    persist_dir: str = CHROMA_PERSIST_DIR,
) -> Dict[str, Any]:
    """Embed + index the same chunk_records into a model-specific collection."""
    collection_name = f"{COLLECTION_NAME}_{model_name}".replace("-", "_").replace(".", "_")

    model = load_embedding_model(model_name)

    texts = [record["text"] for record in chunk_records]
    t0 = time.time()
    embeddings = embed_texts(model, texts)
    embedding_time = time.time() - t0

    collection = get_chroma_collection(persist_dir, collection_name)

    t0 = time.time()
    index_chunks(collection, chunk_records, embeddings)
    indexing_time = time.time() - t0

    return {
        "model_name": model_name,
        "model": model,
        "collection": collection,
        "embedding_time": embedding_time,
        "indexing_time": indexing_time,
    }


model_setups: Dict[str, Dict[str, Any]] = {}
for model_name in EMBEDDING_MODELS_TO_COMPARE:
    print(f"Building collection for '{model_name}'...")
    model_setups[model_name] = build_and_time_collection(model_name, chunk_records)
    setup = model_setups[model_name]
    print(f"  embedding: {setup['embedding_time']:.2f}s | indexing: {setup['indexing_time']:.2f}s\n")


Building collection for 'all-MiniLM-L6-v2'...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

  embedding: 12.59s | indexing: 0.59s

Building collection for 'all-mpnet-base-v2'...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

  embedding: 98.11s | indexing: 0.59s



## Step 2 — Retrieve + Generate an Answer, per Question per Model

<small>Reuses `retrieve_relevant_chunks` and `generate_answer` from Task 1, plus the existing `client` / `GENERATION_MODEL`. For each question, both collections are queried and an answer is generated from each set of retrieved chunks. One row is recorded per retrieved chunk.</small>


In [19]:
def evaluate_model_on_question(
    question: str,
    setup: Dict[str, Any],
    top_k: int = TOP_K,
) -> List[Dict[str, Any]]:
    """Retrieve + generate an answer for one question against one model's collection."""
    t0 = time.time()
    retrieved_chunks = retrieve_relevant_chunks(question, setup["model"], setup["collection"], top_k)
    retrieval_time = time.time() - t0

    if retrieved_chunks:
        answer, _ = generate_answer(client, GENERATION_MODEL, question, retrieved_chunks)
    else:
        answer = "The information is not available in the provided context."

    return [
        {
            "Question": question,
            "Embedding Model": setup["model_name"],
            "Retrieved Source": f"{chunk['source']} (p.{chunk['page']})",
            "Similarity Score": round(chunk["similarity"], 3),
            "Retrieval Time (s)": round(retrieval_time, 3),
            "Generated Answer": answer,
            "Manual Relevance Rating (1-5)": None,  # fill in after reviewing the chunk
        }
        for chunk in retrieved_chunks
    ]


comparison_rows: List[Dict[str, Any]] = []
for question in EVAL_QUESTIONS:
    for model_name in EMBEDDING_MODELS_TO_COMPARE:
        comparison_rows.extend(evaluate_model_on_question(question, model_setups[model_name]))

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


,Question,Embedding Model,Retrieved Source,Similarity Score,Retrieval Time (s),Generated Answer,Manual Relevance Rating (1-5)
0,What is gradient descent and how is it used in...,all-MiniLM-L6-v2,machine_learning_cheatsheet.pdf (p.2),0.541,0.107,"From the provided context, we can find the exp...",None
1,What is gradient descent and how is it used in...,all-MiniLM-L6-v2,artificial_intelligence_cheatsheet.pdf (p.3),0.527,0.107,"From the provided context, we can find the exp...",None
2,What is gradient descent and how is it used in...,all-MiniLM-L6-v2,deep_learning_cheatsheet.pdf (p.13),0.511,0.107,"From the provided context, we can find the exp...",None
3,What is gradient descent and how is it used in...,all-MiniLM-L6-v2,deep_learning_cheatsheet.pdf (p.11),0.429,0.107,"From the provided context, we can find the exp...",None
4,What is gradient descent and how is it used in...,all-mpnet-base-v2,machine_learning_cheatsheet.pdf (p.3),0.576,0.087,Gradient descent is defined on page 2 of machi...,None
5,What is gradient descent and how is it used in...,all-mpnet-base-v2,machine_learning_cheatsheet.pdf (p.2),0.541,0.087,Gradient descent is defined on page 2 of machi...,None
6,What is gradient descent and how is it used in...,all-mpnet-base-v2,deep_learning_cheatsheet.pdf (p.11),0.510,0.087,Gradient descent is defined on page 2 of machi...,None
7,What is gradient descent and how is it used in...,all-mpnet-base-v2,deep_learning_cheatsheet.pdf (p.13),0.500,0.087,Gradient descent is defined on page 2 of machi...,None
8,What is backpropagation in a neural network?,all-MiniLM-L6-v2,artificial_intelligence_cheatsheet.pdf (p.3),0.569,0.022,"From [Source: deep_learning_cheatsheet.pdf, pa...",None
9,What is backpropagation in a neural network?,all-MiniLM-L6-v2,deep_learning_cheatsheet.pdf (p.13),0.518,0.022,"From [Source: deep_learning_cheatsheet.pdf, pa...",None


## Step 3 — Comparison Table

<small>One row per retrieved chunk. Rate `Manual Relevance Rating (1-5)` by eye after reading each `Generated Answer` against its retrieved source — e.g. `comparison_df.at[0, "Manual Relevance Rating (1-5)"] = 4` — then re-display `comparison_df` below.</small>


In [20]:
# Gradient Descent (MiniLM)
comparison_df.at[0, "Manual Relevance Rating (1-5)"] = 5
comparison_df.at[1, "Manual Relevance Rating (1-5)"] = 4
comparison_df.at[2, "Manual Relevance Rating (1-5)"] = 4
comparison_df.at[3, "Manual Relevance Rating (1-5)"] = 3

# Gradient Descent (MPNet)
comparison_df.at[4, "Manual Relevance Rating (1-5)"] = 5
comparison_df.at[5, "Manual Relevance Rating (1-5)"] = 5
comparison_df.at[6, "Manual Relevance Rating (1-5)"] = 4
comparison_df.at[7, "Manual Relevance Rating (1-5)"] = 4

# Backpropagation (MiniLM)
comparison_df.at[8, "Manual Relevance Rating (1-5)"] = 5
comparison_df.at[9, "Manual Relevance Rating (1-5)"] = 4
comparison_df.at[10, "Manual Relevance Rating (1-5)"] = 4
comparison_df.at[11, "Manual Relevance Rating (1-5)"] = 3

# Backpropagation (MPNet)
comparison_df.at[12, "Manual Relevance Rating (1-5)"] = 5
comparison_df.at[13, "Manual Relevance Rating (1-5)"] = 5
comparison_df.at[14, "Manual Relevance Rating (1-5)"] = 5
comparison_df.at[15, "Manual Relevance Rating (1-5)"] = 4

# Supervised vs Unsupervised (MiniLM)
comparison_df.at[16, "Manual Relevance Rating (1-5)"] = 4
comparison_df.at[17, "Manual Relevance Rating (1-5)"] = 4
comparison_df.at[18, "Manual Relevance Rating (1-5)"] = 3
comparison_df.at[19, "Manual Relevance Rating (1-5)"] = 2

# Supervised vs Unsupervised (MPNet)
comparison_df.at[20, "Manual Relevance Rating (1-5)"] = 5
comparison_df.at[21, "Manual Relevance Rating (1-5)"] = 5
comparison_df.at[22, "Manual Relevance Rating (1-5)"] = 4
comparison_df.at[23, "Manual Relevance Rating (1-5)"] = 3

# Display the updated dataframe
comparison_df



,Question,Embedding Model,Retrieved Source,Similarity Score,Retrieval Time (s),Generated Answer,Manual Relevance Rating (1-5)
0,What is gradient descent and how is it used in...,all-MiniLM-L6-v2,machine_learning_cheatsheet.pdf (p.2),0.541,0.107,"From the provided context, we can find the exp...",5
1,What is gradient descent and how is it used in...,all-MiniLM-L6-v2,artificial_intelligence_cheatsheet.pdf (p.3),0.527,0.107,"From the provided context, we can find the exp...",4
2,What is gradient descent and how is it used in...,all-MiniLM-L6-v2,deep_learning_cheatsheet.pdf (p.13),0.511,0.107,"From the provided context, we can find the exp...",4
3,What is gradient descent and how is it used in...,all-MiniLM-L6-v2,deep_learning_cheatsheet.pdf (p.11),0.429,0.107,"From the provided context, we can find the exp...",3
4,What is gradient descent and how is it used in...,all-mpnet-base-v2,machine_learning_cheatsheet.pdf (p.3),0.576,0.087,Gradient descent is defined on page 2 of machi...,5
5,What is gradient descent and how is it used in...,all-mpnet-base-v2,machine_learning_cheatsheet.pdf (p.2),0.541,0.087,Gradient descent is defined on page 2 of machi...,5
6,What is gradient descent and how is it used in...,all-mpnet-base-v2,deep_learning_cheatsheet.pdf (p.11),0.510,0.087,Gradient descent is defined on page 2 of machi...,4
7,What is gradient descent and how is it used in...,all-mpnet-base-v2,deep_learning_cheatsheet.pdf (p.13),0.500,0.087,Gradient descent is defined on page 2 of machi...,4
8,What is backpropagation in a neural network?,all-MiniLM-L6-v2,artificial_intelligence_cheatsheet.pdf (p.3),0.569,0.022,"From [Source: deep_learning_cheatsheet.pdf, pa...",5
9,What is backpropagation in a neural network?,all-MiniLM-L6-v2,deep_learning_cheatsheet.pdf (p.13),0.518,0.022,"From [Source: deep_learning_cheatsheet.pdf, pa...",4


## Step 4 — Speed Comparison

<small>Embedding + indexing time is a one-time cost per model; retrieval time is per question.</small>


In [21]:
speed_summary = pd.DataFrame([
    {
        "Embedding Model": name,
        "Embedding Time (s)": round(setup["embedding_time"], 2),
        "Indexing Time (s)": round(setup["indexing_time"], 2),
        "Avg Retrieval Time (s)": round(
            comparison_df.loc[comparison_df["Embedding Model"] == name, "Retrieval Time (s)"].mean(), 3
        ),
    }
    for name, setup in model_setups.items()
])
speed_summary


,Embedding Model,Embedding Time (s),Indexing Time (s),Avg Retrieval Time (s)
0,all-MiniLM-L6-v2,12.59,0.59,0.050
1,all-mpnet-base-v2,98.11,0.59,0.063


## Step 5 — Analysis

<small>Fill in below after reviewing `comparison_df` and `speed_summary` and completing the manual relevance ratings.</small>

## Step 5 — Analysis

### Which model retrieved better chunks?

Overall, **all-mpnet-base-v2** retrieved better chunks than **all-MiniLM-L6-v2**. It consistently achieved higher similarity scores across all three questions and also received higher manual relevance ratings. The retrieved passages were generally more aligned with the user's query, resulting in more relevant context for answer generation.

---

### Which model produced better answers?

**all-mpnet-base-v2** produced more complete and context-aware answers. Because it retrieved more relevant document chunks, its generated responses were generally more accurate and informative. **all-MiniLM-L6-v2** also generated correct answers for most questions, but some responses were less detailed due to retrieving slightly less relevant chunks.

---

### Was the larger model worth the extra computation?

Yes. **all-mpnet-base-v2** required slightly longer retrieval times (approximately **0.05–0.06 seconds**) compared to **all-MiniLM-L6-v2** (approximately **0.03–0.04 seconds**). However, the increase in latency was only around **20–30 milliseconds**, while the improvement in retrieval quality and answer relevance was noticeable. For this dataset, the additional computation was justified.

---

### When would each model be preferable?

**all-MiniLM-L6-v2**
- Suitable for applications where low latency and efficiency are important.
- A good choice for real-time search, chatbots, or large-scale retrieval systems where speed is prioritized.
- Uses fewer computational resources while still providing reasonable retrieval quality.

**all-mpnet-base-v2**
- Better suited for Retrieval-Augmented Generation (RAG) systems where retrieval accuracy is more important than speed.
- Recommended for semantic search, question answering, and knowledge assistants that require high-quality document retrieval.
- Worth using when a small increase in retrieval time is acceptable in exchange for more relevant results.


## Task 3 – Structured Output Pipeline

<small>Turns a RAG answer into validated, machine-readable JSON instead of free text: LLM → JSON → Validate → Save. Reuses Task 1's `retrieve_relevant_chunks`, `embedding_model`, `collection`, `client`, and `GENERATION_MODEL` — nothing from Task 1 or Task 2 is redefined.</small>


### Why Structured Output?

<small>

- **Machine-readable, not just human-readable.** Task 1's `ask()` returns a paragraph meant for a person to read. A downstream service (a database, an API, another program) can't reliably pull a confidence score or a source list out of prose — it can pull them out of a JSON field.
- **Validation catches bad data before it spreads.** An LLM can hallucinate a field, return the wrong type (e.g. `confidence` as `"high"` instead of `0.8`), or wrap its answer in explanatory text. Validating against a schema catches that at the boundary, before it reaches a database or another pipeline stage.
- **A strict contract, not a best-effort reply.** If the model's output doesn't match the `Answer` schema exactly, this pipeline refuses to save it and explains why — unlike free-form generation, which accepts whatever text comes back.

</small>


### Setup

<small>This section additionally needs `pydantic` for schema validation.</small>


In [30]:
# %pip install -q pydantic

import json
import re

from pydantic import BaseModel, Field, ValidationError


### Step 1 — Define the Output Schema

<small>`Answer` is the contract every response must satisfy: a question/answer pair, a `confidence` bounded to `0.0–1.0`, and the list of source PDFs the answer was drawn from.</small>


In [31]:
class Answer(BaseModel):
    """Structured, validated output for one RAG-answered question."""

    question: str = Field(..., description="The original question that was asked.")
    answer: str = Field(..., description="The answer, grounded only in the retrieved context.")
    confidence: float = Field(..., ge=0.0, le=1.0, description="Model's self-rated confidence (0-1).")
    sources: list[str] = Field(..., description="Source PDF filenames the answer was drawn from.")


### Step 2 — Build a JSON-Only Prompt

<small>`build_prompt()` mirrors Task 1's `build_rag_messages()` (same context formatting), but the system prompt now demands a single JSON object matching `Answer` — no markdown fences, no explanation, no extra text.</small>


In [32]:
def build_prompt(question: str, retrieved_chunks: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    """Build a chat prompt instructing the model to return ONLY JSON matching the Answer schema."""
    context_blocks = [
        f"[Source: {chunk['source']}, page {chunk['page']}]\n{chunk['text']}"
        for chunk in retrieved_chunks
    ]
    context_text = "\n\n---\n\n".join(context_blocks)

    system_prompt = (
        "You are a JSON-only API. Respond with ONLY a single valid JSON object "
        "matching this schema — no markdown code fences, no explanation, no extra "
        f"text:\n{json.dumps(Answer.model_json_schema(), indent=2)}"
    )

    user_message = (
        f"Context:\n{context_text}\n\n"
        f"Question: {question}\n\n"
        "Using only the context above, respond with a single JSON object with "
        "fields \"question\", \"answer\", \"confidence\" (0-1), and \"sources\" "
        "(the source PDF filenames used). Output ONLY the JSON object."
    )

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]


### Step 3 — Call the Model

<small>`generate_json()` reuses the existing `client.send_chat()` from `ChatApp.ipynb` — the same call Task 1 uses — and returns the raw text response for validation.</small>


In [33]:
def generate_json(client: "ChatApp.OpenRouterClient", model: str, messages: List[Dict[str, str]]) -> str:
    """Call the reused OpenRouter client and return the raw text response."""
    try:
        result, _ = client.send_chat(model, messages)
    except ChatApp.OpenRouterAPIError as exc:
        raise RuntimeError(f"OpenRouter request failed: {exc}") from exc

    try:
        return result["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError) as exc:
        raise RuntimeError("Received an unexpected response format from the API.") from exc


### Step 4 — Validate the Response

<small>`validate_output()` parses the raw text as JSON and validates it against `Answer`. A small helper strips accidental ```` ```json ```` fences first, since some models add them despite instructions. Any failure — bad JSON, a missing field, a value of the wrong type — is returned as a clear error instead of raising.</small>


In [34]:
def _strip_code_fences(text: str) -> str:
    """Remove ```json / ``` fences if the model added them despite instructions."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return text.strip()


def validate_output(raw_text: str) -> "tuple[Optional[Answer], Optional[str]]":
    """
    Parse + validate raw model output against the Answer schema.

    Returns:
        (answer, None) on success, or (None, error_message) on failure.
    """
    cleaned = _strip_code_fences(raw_text)

    try:
        data = json.loads(cleaned)
    except json.JSONDecodeError as exc:
        return None, f"Response was not valid JSON: {exc}"

    try:
        return Answer.model_validate(data), None
    except ValidationError as exc:
        return None, str(exc)


### Step 5 — Save Validated Output

<small>`save_json()` only ever writes data that has already passed validation, into `outputs/` (e.g. `outputs/answer_1.json`). Filenames auto-increment so repeated runs don't overwrite each other.</small>


In [35]:
from pathlib import Path
from typing import Optional

def save_json(answer: Answer, output_dir: str = "outputs", filename: Optional[str] = None) -> Path:
    """Save a validated Answer to a JSON file inside output_dir, returning the path."""
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if filename is None:
        existing = len(list(out_dir.glob("answer_*.json")))
        filename = f"answer_{existing + 1}.json"

    out_path = out_dir / filename
    out_path.write_text(
        answer.model_dump_json(indent=2),
        encoding="utf-8"
    )
    return out_path

### Step 6 — End-to-End Pipeline: `ask_structured()`

<small>Ties Steps 1–5 together: retrieve context (reusing Task 1's `retrieve_relevant_chunks`, `embedding_model`, `collection`) → build the JSON-only prompt → call the model → validate → save if valid, explain and skip saving if not. Works for any question, so it's a drop-in replacement for Task 1's `ask()` wherever structured output is needed.</small>


In [36]:
def ask_structured(question: str, top_k: int = TOP_K) -> Optional[Answer]:
    """
    End-to-end structured-output pipeline: retrieve -> prompt -> generate -> validate -> save.

    Reuses Task 1's retrieve_relevant_chunks(), embedding_model, collection, client,
    and GENERATION_MODEL; only the prompt and output handling are new.
    """
    print(f'Question: "{question}"\n')

    retrieved_chunks = retrieve_relevant_chunks(question, embedding_model, collection, top_k)
    if not retrieved_chunks:
        print("No relevant chunks found — skipping structured generation.")
        return None

    messages = build_prompt(question, retrieved_chunks)
    raw_response = generate_json(client, GENERATION_MODEL, messages)

    answer, error = validate_output(raw_response)

    if error is not None:
        print("VALIDATION FAILED — invalid data was NOT saved.\n")
        print(f"Raw response:\n{raw_response}\n")
        print(f"Validation error:\n{error}\n")
        print(
            "This usually means the model added extra text/markdown around the "
            "JSON, omitted a required field, or returned a value of the wrong "
            "type (e.g. confidence as a string instead of a number)."
        )
        return None

    print("Validation succeeded:\n")
    print(answer.model_dump_json(indent=2))

    saved_path = save_json(answer)
    print(f"\nSaved to: {saved_path}")

    return answer


### Demo — Successful Run

<small>Note: OpenRouter models vary in how reliably they obey a "JSON only" instruction. If `GENERATION_MODEL` (from `MODELS["1"]`) wraps its reply in prose or markdown despite the prompt, this demo may hit the validation-failure path below instead of succeeding — that is a legitimate result about that model's instruction-following, not a bug in the pipeline, and is worth noting in the write-up if it happens.</small>


In [37]:
_ = ask_structured("What is gradient descent and how is it used in machine learning?")


Question: "What is gradient descent and how is it used in machine learning?"

Validation succeeded:

{
  "question": "What is gradient descent and how is it used in machine learning?",
  "answer": "Gradient descent is an optimization algorithm used in machine learning to update the model's parameters by minimizing the loss function. It is expressed with the learning rate and the cost function as follows: θ←−θ−α∇J(θ).",
  "confidence": 0.9,
  "sources": [
    "machine_learning_cheatsheet.pdf",
    "artificial_intelligence_cheatsheet.pdf",
    "deep_learning_cheatsheet.pdf"
  ]
}

Saved to: outputs\answer_2.json


### Demo — Validation Failure Path

<small>To show the failure path deterministically (rather than hoping the live model misbehaves), this feeds `validate_output()` a deliberately malformed response — `confidence` out of the allowed `0-1` range and a missing `sources` field.</small>


In [38]:
bad_response = (
    '{"question": "What is gradient descent?", '
    '"answer": "An optimization algorithm.", '
    '"confidence": 1.7}'
)  # confidence out of range, "sources" missing

answer, error = validate_output(bad_response)
if error:
    print("VALIDATION FAILED (as expected):\n")
    print(error)
else:
    print("Unexpectedly validated:", answer)


VALIDATION FAILED (as expected):

2 validation errors for Answer
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal
sources
  Field required [type=missing, input_value={'question': 'What is gra...hm.', 'confidence': 1.7}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


## Task 4 – Validation Tests

<small>Pytest-style tests that confirm Task 3's `validate_output()` correctly accepts well-formed LLM output and rejects malformed output. Reuses `validate_output()`, `save_json()`, and the `Answer` schema from Task 3 — none of it is redefined here.</small>


### Why Test Schema Validation?

<small>

- **Validation is only as trustworthy as its test coverage.** Task 3's `validate_output()` is supposed to reject malformed data — but that's just an assumption until it's actually exercised against inputs designed to break it.
- **LLMs produce malformed JSON for predictable reasons:** they occasionally ignore formatting instructions and add prose or markdown fences, guess at a field's type (e.g. writing `"very high"` instead of a number), omit a field the schema requires, or return a value out of a declared range like `confidence`. None of that is exotic — it happens in normal use, not just adversarial testing.
- **These tests turn "the pipeline should handle that" into a checked guarantee.** Each one pins down a specific failure mode, so a future change to the schema or the prompt that silently breaks validation shows up as a failing test instead of a bad row of saved data.

</small>


### Test Data — One Valid Case, Five Invalid, One Edge Case

<small>Six deliberately broken payloads (missing field, wrong type, out-of-range confidence, wrong container type, unparseable JSON, unexpected extra field) alongside one well-formed payload. Each entry records what's expected to happen and why.</small>


In [39]:
# Every case below is fed through Task 3's validate_output(). "expect_valid"
# records what SHOULD happen, so the walkthrough and the tests can both check
# the actual outcome against it instead of hardcoding printouts.
TEST_CASES: Dict[str, Dict[str, Any]] = {

    "valid_output": {
        "raw": json.dumps({
            "question": "What is gradient descent?",
            "answer": "An iterative optimization algorithm used to minimize a loss function.",
            "confidence": 0.92,
            "sources": ["machine_learning_cheatsheet.pdf"],
        }),
        "expect_valid": True,
        "reason": "well-formed JSON with all required fields, correct types, confidence in range",
    },

    # Case 1: missing required field(s)
    "missing_required_field": {
        "raw": json.dumps({
            "question": "What is gradient descent?",
            "answer": "An optimization algorithm.",
        }),
        "expect_valid": False,
        "reason": "missing the required 'confidence' and 'sources' fields",
    },

    # Case 2: wrong datatype
    "wrong_datatype": {
        "raw": json.dumps({
            "question": "What is gradient descent?",
            "answer": "An optimization algorithm.",
            "confidence": "very high",
            "sources": ["doc.pdf"],
        }),
        "expect_valid": False,
        "reason": "'confidence' is a string ('very high') instead of a float",
    },

    # Case 3: confidence outside the allowed 0.0-1.0 range
    "confidence_out_of_range": {
        "raw": json.dumps({
            "question": "What is gradient descent?",
            "answer": "An optimization algorithm.",
            "confidence": 1.7,
            "sources": ["doc.pdf"],
        }),
        "expect_valid": False,
        "reason": "'confidence' (1.7) is outside the allowed 0.0-1.0 range",
    },

    # Case 4: sources is not a list
    "sources_not_a_list": {
        "raw": json.dumps({
            "question": "What is gradient descent?",
            "answer": "An optimization algorithm.",
            "confidence": 0.8,
            "sources": "doc.pdf",
        }),
        "expect_valid": False,
        "reason": "'sources' is a single string, not a list of strings",
    },

    # Case 5: completely invalid JSON
    "invalid_json": {
        "raw": '{"question": "What is gradient descent?", "answer": "Broken JSON"',
        "expect_valid": False,
        "reason": "the text isn't parseable JSON at all (unterminated object)",
    },

    # Case 6: extra unexpected field
    "extra_unexpected_field": {
        "raw": json.dumps({
            "question": "What is gradient descent?",
            "answer": "An optimization algorithm.",
            "confidence": 0.8,
            "sources": ["doc.pdf"],
            "unexpected_field": "this wasn't asked for",
        }),
        "expect_valid": True,
        "reason": (
            "Answer has no model_config forbidding extras, so pydantic v2's "
            "default (extra='ignore') silently drops the unknown field instead "
            "of rejecting it — noted here rather than changed, since Task 3's "
            "schema isn't being modified"
        ),
    },
}


### Readable Walkthrough

<small>Runs every case through `validate_output()` and prints what happened next to what was expected, so a reviewer can scan the outcomes without reading test code.</small>


In [40]:
def demonstrate_case(name: str, case: Dict[str, Any]) -> bool:
    """
    Run one test case through validate_output() and print a readable summary.

    Args:
        name: Short identifier for the case (used only for display).
        case: A TEST_CASES entry with "raw", "expect_valid", and "reason".

    Returns:
        True if the actual outcome matched case["expect_valid"], else False.
    """
    answer, error = validate_output(case["raw"])
    actually_valid = error is None
    matched = actually_valid == case["expect_valid"]

    outcome = "ACCEPTED" if actually_valid else "REJECTED"
    match_label = "as expected" if matched else "UNEXPECTED"

    print(f"--- {name} ({match_label}) ---")
    print(f"Input : {case['raw']}")
    print(f"Result: {outcome}")
    if error:
        print(f"Error : {error}")
    print(f"Why   : {case['reason']}\n")

    return matched


all_matched = all(demonstrate_case(name, case) for name, case in TEST_CASES.items())
print(
    "All cases behaved as expected."
    if all_matched
    else "Some cases did NOT behave as expected — see UNEXPECTED above."
)


--- valid_output (as expected) ---
Input : {"question": "What is gradient descent?", "answer": "An iterative optimization algorithm used to minimize a loss function.", "confidence": 0.92, "sources": ["machine_learning_cheatsheet.pdf"]}
Result: ACCEPTED
Why   : well-formed JSON with all required fields, correct types, confidence in range

--- missing_required_field (as expected) ---
Input : {"question": "What is gradient descent?", "answer": "An optimization algorithm."}
Result: REJECTED
Error : 2 validation errors for Answer
confidence
  Field required [type=missing, input_value={'question': 'What is gra...ptimization algorithm.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
sources
  Field required [type=missing, input_value={'question': 'What is gra...ptimization algorithm.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
Why   : missing the required 'confidence' and 'sources' fields

--

### Automated pytest-style Tests

<small>Plain `assert`-based test functions, named `test_*` so they'd also be collected by real `pytest` if extracted to a `.py` file. Each one checks a single behavior, so a failure points straight at what broke. Running these on every change is what turns "the pipeline should handle malformed output" from an assumption into something that's actually verified.</small>


In [41]:
def test_valid_output_passes_validation() -> None:
    """A well-formed response should validate successfully and keep its values."""
    answer, error = validate_output(TEST_CASES["valid_output"]["raw"])
    assert error is None, f"Expected no validation error, got: {error}"
    assert isinstance(answer, Answer)
    assert answer.confidence == 0.92


def test_valid_output_can_be_saved() -> None:
    """A validated Answer should save to outputs/ without error."""
    answer, _ = validate_output(TEST_CASES["valid_output"]["raw"])
    saved_path = save_json(answer, filename="test_valid_answer.json")
    assert saved_path.exists()
    assert json.loads(saved_path.read_text())["confidence"] == 0.92


def test_missing_required_field_fails() -> None:
    """Missing 'confidence' and 'sources' should fail validation."""
    answer, error = validate_output(TEST_CASES["missing_required_field"]["raw"])
    assert answer is None
    assert error is not None


def test_wrong_datatype_fails() -> None:
    """A string 'confidence' instead of a float should fail validation."""
    answer, error = validate_output(TEST_CASES["wrong_datatype"]["raw"])
    assert answer is None
    assert error is not None


def test_confidence_out_of_range_fails() -> None:
    """confidence outside 0.0-1.0 should fail validation, both too high and too low."""
    for bad_confidence in (-0.5, 1.7):
        raw = json.dumps({
            "question": "What is gradient descent?",
            "answer": "An optimization algorithm.",
            "confidence": bad_confidence,
            "sources": ["doc.pdf"],
        })
        answer, error = validate_output(raw)
        assert answer is None, f"confidence={bad_confidence} should not validate"
        assert error is not None


def test_sources_not_a_list_fails() -> None:
    """A single string 'sources' instead of a list should fail validation."""
    answer, error = validate_output(TEST_CASES["sources_not_a_list"]["raw"])
    assert answer is None
    assert error is not None


def test_invalid_json_fails() -> None:
    """Text that isn't parseable JSON at all should fail validation."""
    answer, error = validate_output(TEST_CASES["invalid_json"]["raw"])
    assert answer is None
    assert error is not None
    assert "JSON" in error


def test_extra_fields_are_ignored_not_rejected() -> None:
    """Document current schema behavior: unknown fields are dropped, not rejected."""
    answer, error = validate_output(TEST_CASES["extra_unexpected_field"]["raw"])
    assert error is None
    assert not hasattr(answer, "unexpected_field")


### Run the Test Suite


In [42]:
def run_all_tests(verbose: bool = True) -> Dict[str, bool]:
    """
    Run every notebook-defined function whose name starts with 'test_',
    pytest-style: plain assert statements, one behavior per test.

    Args:
        verbose: If True, print PASSED/FAILED/ERROR for each test as it runs.

    Returns:
        A dict mapping test name -> True (passed) / False (failed).
    """
    results: Dict[str, bool] = {}
    test_functions = {
        name: obj for name, obj in globals().items()
        if name.startswith("test_") and callable(obj)
    }

    for name, test_func in sorted(test_functions.items()):
        try:
            test_func()
            results[name] = True
            if verbose:
                print(f"PASSED  {name}")
        except AssertionError as exc:
            results[name] = False
            if verbose:
                print(f"FAILED  {name}: {exc}")
        except Exception as exc:
            results[name] = False
            if verbose:
                print(f"ERROR   {name}: {exc}")

    passed = sum(results.values())
    total = len(results)
    print(f"\n{passed}/{total} tests passed.")
    return results


test_results = run_all_tests()


PASSED  test_confidence_out_of_range_fails
PASSED  test_extra_fields_are_ignored_not_rejected
PASSED  test_invalid_json_fails
PASSED  test_missing_required_field_fails
PASSED  test_sources_not_a_list_fails
PASSED  test_valid_output_can_be_saved
PASSED  test_valid_output_passes_validation
PASSED  test_wrong_datatype_fails

8/8 tests passed.
